In [ ]:
# # Blender + Qwen-Image-Edit-2509 smoke (Kaggle T4 16GB, GGUF Q4_K_S)
#
# Pipeline #2 (Apache-2.0): Blender 3D → Qwen instruction edit (whole frame, vehicle kept by instruction) → composite (freeze vehicle pixel-perfect) → albumentations polish.
#
# ⚠️ Qwen-Image-Edit-2509 = 20B params. У bf16 = 40GB → NЕ влазить у T4 16GB. Тому **обов'язково GGUF Q4_K_S** (~12 GB) + `enable_sequential_cpu_offload()`.
#
# Vehicle pixels гарантовано замороженi Stage 4 composite (binary mask freeze). Qwen НЕ має mask — це instruction edit; інструкція «лиши техніку» + Stage 4 — спільно гарантують piксель-perfect збереження.
#
# ## Передумови
# 1. Kaggle dataset `dariachuprina/yolo-bluebird-df-assets` attached як Input.
# 2. Accelerator: **GPU T4 x2** (Settings → Accelerator). P100 більше не доступний (Kaggle drop sm_60 у нових torch).
# 3. Internet: **On**.
# 4. **HF_TOKEN** — Add-ons → Secrets → Add HF_TOKEN.

In [ ]:
# # ⚙️ SETTINGS — edit here, then re-run
#
# Усі ручки в одному місці. Не треба чіпати YAML.
#
# - Змінила **steps / guidance / true_cfg / inference_size / prompts / polish** → re-run цей селл, потім Cell 10 (Stage 3+4+5). Без reload pipeline — швидко.
# - Змінила **gguf_url / base_model / force_precision** → re-run цей селл + **Cell 9 (Load)**.
# - Змінила **scene / road / landscapes** → re-render: re-run цей селл + **Cell 7 (Stage 1)** + далі.
#
# ⚠️ `true_cfg_scale > 1.0` ≈ ×2 час inference, але без нього negative_prompt ігнорується.

In [ ]:
# ⚙️ SETTINGS — РЕДАГУЙ ТУТ (pipeline #2: Qwen-Image-Edit-2509, Apache-2.0)
SETTINGS = {
    'model': 'qwen_edit',
    'base_model': 'Qwen/Qwen-Image-Edit-2509',
    'pipeline': 'QwenImageEditPlusPipeline',
    # T4 16GB — 768² безпечно (≈12 GB GGUF + 3 GB activations). 1024² можливо OOM.
    'inference_size': [768, 768],
    'steps': 25,                # smoke; фінал 35
    'guidance': 1.0,            # Qwen guidance_scale (real CFG несе true_cfg_scale)
    'true_cfg_scale': 4.0,      # 2-8; without it negative ignored
    # GGUF transformer — ОБОВ'ЯЗКОВО для T4. Q4_K_S 12.2 GB (Q4_K_M 13.1 GB впритул).
    'gguf_url': 'https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/resolve/main/Qwen-Image-Edit-2509-Q4_K_S.gguf',
    'force_precision': 'gguf_offload',   # явно (auto-detect теж дасть gguf_offload на T4)
    # depth-ControlNet: diffusers Qwen-depth API нестабільне → off за замовч.
    'depth_control_enabled': False, 'depth_control_scale': 0.9,

    # --- relight: підтягує яскравість/тон ТЕХНІКИ під фон (bbox не чіпає) ---
    'relight_enabled': True,
    'relight_strength': 0.55,
    'relight_match_color': True,
    # --- scene (зміна → re-render через Stage 1) ---
    'road_under_vehicle': True,
    'landscapes': ['dirt_road', 'field', 'dirt_road', 'forest_belt'],
}

# ─────────────────────────────────────────────────────────────────────────
# PROMPT — редагуй текст напряму. {плейсхолдери} підставляються з metadata кадру.
PROMPT_TEMPLATE = (
    'aerial drone reconnaissance photo of a {landscape} in {season}, {weather}, '
    '{sun_cardinal} sunlight at {sun_elevation_deg:.0f} degrees elevation, '
    'shot from {altitude_m:.0f} meters altitude at {view_angle_deg:.0f} degrees '
    'oblique top-down angle'
)
LANDSCAPE_CUES = {
    'dirt_road': ('the vehicle is driving along a narrow dirt road, two parallel '
                  'tire ruts in packed earth running directly beneath and ahead of '
                  'the vehicle, dusty unpaved track'),
    'field': ('the vehicle sits on a faint farm track crossing an open field, '
              'flattened grass and tire marks under the vehicle'),
    'forest_belt': ('the vehicle is on a dirt track beside a tree line / forest '
                    'shelterbelt, track running under the vehicle'),
}
SCALE_CUE = ('seen straight from above at high altitude, everything at true aerial '
             'scale, vegetation appears as tiny fine texture, no large foreground objects')
CAMERA_CUE = ('low-resolution aerial surveillance footage, grainy telephoto drone '
              'shot, overcast flat lighting, slightly soft focus, visible sensor noise '
              'and jpeg compression artifacts, muted desaturated colors, candid '
              'unedited photo, amateur photo')
NEGATIVE_PROMPT = (
    'person, soldier, vehicle, tank, truck, car, motorcycle, building, road sign, '
    'text, watermark, ui, 3d render, cgi, render, video game, unreal engine, octane, '
    'blender, plastic, glossy, waxy, airbrushed, cinematic, dramatic lighting, '
    'golden hour, lens flare, bokeh, depth of field, hdr, oversaturated, vivid, '
    'overprocessed, masterpiece, artstation, oversharpened, illustration, painting, '
    'blurry, low quality'
)

# ─────────────────────────────────────────────────────────────────────────
# STAGE 5 POLISH — albumentations «шліфування поверх» (туман/зерно/шум/jpeg/blur).
# pixel-only → bbox НЕ рухається. false щоб вимкнути блок; p=ймовірність.
POLISH = {
    'enabled': True,
    'iso_noise':           {'p': 0.7,  'color_shift': [0.01, 0.05], 'intensity': [0.1, 0.5]},
    'gauss_noise':         {'p': 0.35, 'std_range': [0.02, 0.07]},
    'motion_blur':         {'p': 0.3,  'blur_limit': [3, 7]},
    'defocus':             {'p': 0.15, 'radius': [1, 3]},
    'fog':                 {'p': 0.25, 'fog_coef_range': [0.05, 0.25], 'alpha_coef': 0.08},
    'brightness_contrast': {'p': 0.5,  'brightness': 0.12, 'contrast': 0.12},
    'chromatic':           {'p': 0.3,  'primary': 0.02, 'secondary': 0.01},
    'downscale':           {'p': 0.2,  'scale_range': [0.6, 0.9]},
    'jpeg':                {'p': 0.9,  'quality': [60, 85]},
    'seed_offset': 9000,
}

# ─────────────────────────────────────────────────────────────────────────
def _diffusion_overrides():
    return {'enabled': True, 'base_model': SETTINGS['base_model'],
            'pipeline': SETTINGS['pipeline'], 'inference_size': SETTINGS['inference_size'],
            'steps': SETTINGS['steps'], 'guidance': SETTINGS['guidance'],
            'true_cfg_scale': SETTINGS['true_cfg_scale'],
            'force_precision': SETTINGS['force_precision'],
            'depth_control': {'enabled': SETTINGS['depth_control_enabled'],
                              'scale': SETTINGS['depth_control_scale']},
            'gguf': {'url': SETTINGS['gguf_url']},
            'relight': {'enabled': SETTINGS['relight_enabled'],
                        'strength': SETTINGS['relight_strength'],
                        'match_color': SETTINGS['relight_match_color']},
            'prompt_template': PROMPT_TEMPLATE, 'negative_prompt': NEGATIVE_PROMPT,
            'landscape_cues': LANDSCAPE_CUES, 'scale_cue': SCALE_CUE, 'camera_cue': CAMERA_CUE}

def apply_settings(cfg_dict):
    cfg_dict.setdefault('diffusion', {}).update(_diffusion_overrides())
    sc = cfg_dict.setdefault('scene', {})
    sc['road_under_vehicle'] = SETTINGS['road_under_vehicle']
    sc['landscapes'] = SETTINGS['landscapes']
    cfg_dict['polish'] = dict(POLISH)
    return cfg_dict

polish_cfg = dict(POLISH)
try:
    diff_cfg.update(_diffusion_overrides())
    print('✓ diff_cfg + polish_cfg оновлено — re-run Cell 10 (Stage 3+4+5)')
except NameError:
    print(f"✓ settings задані (model={SETTINGS['model']}) — далі Cell 3 → 4 → 5 …")


In [ ]:
# Cell 3 — env check + HF_TOKEN
#
# HF_TOKEN — три шляхи (один з них):
#   (A) Kaggle Secrets: Add-ons → Secrets → Add HF_TOKEN → Attach
#   (B) Variables sidebar → HF_TOKEN = hf_xxx
#   (C) Inline paste нижче (раскоментуй), run, але НЕ Save Version
import os, sys, torch

# (C) inline fallback:
# os.environ['HF_TOKEN'] = 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

if 'HF_TOKEN' not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        raise RuntimeError(
            f'HF_TOKEN не set. Add-ons→Secrets→Attach HF_TOKEN, або uncomment Cell 3. '
            f'Kaggle secret error: {e}'
        )

print(f'Python: {sys.version.split()[0]}')
print(f'CUDA: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'enable GPU T4 x2 у Settings → Accelerator'
caps = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'  sm_{caps[0]}{caps[1]} | VRAM {vram_gb:.1f} GB | CUDA {torch.version.cuda}')
print(f'  bf16 native: {caps >= (8, 0)} (T4 sm_75 → no, GGUF compute will be emulated, повільніше)')
print(f"HF_TOKEN: {os.environ['HF_TOKEN'][:8]}...")

In [ ]:
# Cell 4 — install diffusion + render stack (+ GGUF support)
# diffusers >=0.36 для QwenImageEditPlusPipeline, gguf для quantization.
!pip install -q 'diffusers>=0.36' 'transformers>=4.46' accelerate sentencepiece safetensors
!pip install -q gguf  # GGUFQuantizationConfig support
!pip install -q blenderproc h5py pyyaml opencv-python-headless albumentations
print('✓ deps installed')

In [ ]:
# Cell 5 — clone repo (public) + auto-detect assets + import dispatch
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path('/kaggle/working/yolo-bluebierd')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone',
                'https://github.com/ChuprinaDaria/YOLO-Bluebierd.git',
                str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Assets — auto-detect mount path (manual Input vs CLI dataset_sources різний)
INPUT = Path('/kaggle/input')
matches = list(INPUT.rglob('hdri'))
assert matches, 'attach dataset yolo-bluebird-df-assets як Input (+ Add Input)'
ASSETS = matches[0].parent
print(f'repo:   {REPO_DIR}')
print(f'assets: {ASSETS}')
for sub in ('hdri', 'textures/ground', 'models/light_vehicle'):
    n = len(list((ASSETS / sub).rglob('*')))
    print(f'  {sub}: {n} items')

# Dispatch loader (qwen_edit pipeline)
from datasetforge.pipelines.qwen_edit import load_model as _m
load_pipeline, stage3_fn = _m.load_pipeline, _m.edit_one
from datasetforge.pipelines.shared import composite, polish
print(f'✓ qwen_edit module imported (stage3={stage3_fn.__name__})')

In [ ]:
# Cell 6 — BlenderProc quickstart (downloads Blender ~330 MB)
!rm -rf /root/blender 2>/dev/null
!blenderproc quickstart 2>&1 | tail -5
print('\n✓ Blender ready')

In [ ]:
# Cell 7 — Stage 1: render 1 frame at diffusion resolution
import os, subprocess, yaml
from pathlib import Path

REPO_DIR = Path('/kaggle/working/yolo-bluebierd')
ASSETS = Path([p for p in Path('/kaggle/input').rglob('hdri')][0].parent)
CFG_SRC = REPO_DIR / 'datasetforge/configs/v1_light_vehicle.yaml'
OUT = Path('/kaggle/working/v2_qwen_smoke_1frame')
OUT.mkdir(parents=True, exist_ok=True)

cfg_dict = yaml.safe_load(CFG_SRC.read_text())
cfg_dict['image_size'] = SETTINGS['inference_size']
cfg_dict = apply_settings(cfg_dict)
CFG_RUN = OUT / 'v1_light_vehicle_qwen.yaml'
CFG_RUN.write_text(yaml.safe_dump(cfg_dict))
print(f'render @ {SETTINGS["inference_size"]}, cfg → {CFG_RUN}')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
cmd = ['blenderproc', 'run', str(REPO_DIR / 'datasetforge/engine/render_runner.py'),
       '--config', str(CFG_RUN), '--n', '1', '--out', str(OUT),
       '--assets-root', str(ASSETS), '--seed', '42']
subprocess.run(cmd, env=env, check=True)

print(f'\n✓ Stage 1 → {OUT}')
for sub in ['images','labels','depth','normals','vehicle_masks','metadata']:
    n = len(list((OUT / sub).glob('*')))
    print(f'  {sub}: {n} file(s)')

In [ ]:
# Cell 8 — Stage 1 preview (RGB+bbox / depth / normals / mask)
import json, cv2, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

OUT = Path('/kaggle/working/v2_qwen_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
print(f'preview: {stem}')

rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
depth = cv2.imread(str(OUT/'depth'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
normals_raw = cv2.imread(str(OUT/'normals'/f'{stem}.png'), cv2.IMREAD_UNCHANGED)
mask = cv2.imread(str(OUT/'vehicle_masks'/f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
meta = json.loads((OUT/'metadata'/f'{stem}.json').read_text())

H, W = rgb.shape[:2]
rgb_pil = Image.fromarray(rgb).copy()
d = ImageDraw.Draw(rgb_pil)
lbl = OUT/'labels'/f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        p = line.split(); xc, yc, w, h = map(float, p[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=3)

normals_viz = ((normals_raw.astype(np.float32)/65535.0)*255).astype(np.uint8) if normals_raw.dtype==np.uint16 else normals_raw

fig, ax = plt.subplots(2, 2, figsize=(12, 12))
ax[0,0].imshow(rgb_pil); ax[0,0].set_title('RGB + bbox'); ax[0,0].axis('off')
ax[0,1].imshow(depth, cmap='viridis'); ax[0,1].set_title(f'Depth max={int(depth.max())} mm'); ax[0,1].axis('off')
ax[1,0].imshow(normals_viz); ax[1,0].set_title('Normals'); ax[1,0].axis('off')
ax[1,1].imshow(mask, cmap='gray'); ax[1,1].set_title(f'Mask veh_px={int((mask>=128).sum())}'); ax[1,1].axis('off')
plt.tight_layout(); plt.show()

print(f"\nsun: {meta['sun_cardinal']} @ {meta['sun_elevation_deg']:.0f}° elev")
print(f"season={meta['season']} weather={meta['weather']} landscape={meta['landscape']}")
print(f"distance={meta['distance_m']}m angle={meta['view_angle_deg']}°")

In [ ]:
# Cell 9 — Load Qwen-Image-Edit-2509 GGUF Q4_K_S + sequential CPU offload.
# Перший запуск: GGUF ~12 GB stream download + T5/CLIP/VAE ~3 GB → 15-25 min на T4.
# Кеш у /root/.cache/huggingface (per-session, не persistent у Kaggle).
import time, yaml, torch
from pathlib import Path

CFG_RUN = Path('/kaggle/working/v2_qwen_smoke_1frame/v1_light_vehicle_qwen.yaml')
cfg_full = yaml.safe_load(CFG_RUN.read_text())
diff_cfg = cfg_full['diffusion']
polish_cfg = cfg_full.get('polish', polish_cfg)
print(f"model={SETTINGS['model']} | pipeline={diff_cfg.get('pipeline')}")
print(f"base={diff_cfg['base_model']}")
print(f"gguf={diff_cfg['gguf']['url'].split('/')[-1]}")
print(f"size={diff_cfg['inference_size']} steps={diff_cfg['steps']} true_cfg={diff_cfg.get('true_cfg_scale')}")
print(f"relight={diff_cfg.get('relight', {})}")
print(f"polish={'on' if polish_cfg.get('enabled') else 'off'}")

t0 = time.time()
pipe = load_pipeline(diff_cfg, device='cuda')
try:
    torch.cuda.synchronize()
except Exception:
    pass
print(f'\n✓ loaded in {time.time()-t0:.0f}s')
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (peak очікується ~14 GB)")

In [ ]:
# Cell 10 — Stage 3 (Qwen edit) + Stage 4 (composite, freeze vehicle) + Stage 5 (polish) + 4-up preview
# T4 + GGUF Q4 + sequential offload: ~3-6 min на кадр (Qwen 20B повільний навіть квантизований)
import time, json, numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

OUT = Path('/kaggle/working/v2_qwen_smoke_1frame')
stem = sorted(p.stem for p in (OUT/'images').glob('*.jpg'))[0]
for sub in ('ai_bg', 'composite', 'final'):
    (OUT/sub).mkdir(exist_ok=True)

# Stage 3 — Qwen instruction edit (whole frame; mask ігнорується)
t0 = time.time()
sidecar = stage3_fn(
    pipe,
    rgb_path=OUT/'images'/f'{stem}.jpg',
    depth_path=OUT/'depth'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json',
    out_path=OUT/'ai_bg'/f'{stem}.png',
    diffusion_cfg=diff_cfg,
)
t_edit = time.time() - t0
print(f'✓ Stage 3 ({stage3_fn.__name__}) {t_edit:.0f}s | seed={sidecar["seed"]}')
print(f'prompt: {sidecar["prompt"][:160]}...')

# Stage 4 — composite (freeze vehicle pixels by binary mask)
stats = composite.composite_one(
    rgb_path=OUT/'images'/f'{stem}.jpg', ai_bg_path=OUT/'ai_bg'/f'{stem}.png',
    mask_path=OUT/'vehicle_masks'/f'{stem}.png', normals_path=OUT/'normals'/f'{stem}.png',
    meta_path=OUT/'metadata'/f'{stem}.json', out_path=OUT/'composite'/f'{stem}.png',
    diffusion_cfg=diff_cfg, assert_pixel_identity=True)
print(f'✓ Stage 4 composite {stats.get("relight", {})}')

# Stage 5 — polish (albumentations pixel-only; bbox не рухається)
_seed = int(json.loads((OUT/'metadata'/f'{stem}.json').read_text()).get('seed', 0)) + int(polish_cfg.get('seed_offset', 9000))
pstats = polish.polish_one(OUT/'composite'/f'{stem}.png', OUT/'final'/f'{stem}.png', polish_cfg, seed=_seed)
print(f'✓ Stage 5 polish {pstats["transforms"]}')

# 4-up preview: raw → ai_bg → composite → final+bbox
rgb = np.array(Image.open(OUT/'images'/f'{stem}.jpg'))
ai_bg = np.array(Image.open(OUT/'ai_bg'/f'{stem}.png'))
comp = np.array(Image.open(OUT/'composite'/f'{stem}.png'))
final = np.array(Image.open(OUT/'final'/f'{stem}.png'))
H, W = final.shape[:2]
final_pil = Image.fromarray(final).copy()
d = ImageDraw.Draw(final_pil)
lbl = OUT/'labels'/f'{stem}.txt'
if lbl.exists():
    for line in lbl.read_text().strip().splitlines():
        pp = line.split(); xc, yc, w, h = map(float, pp[1:5])
        d.rectangle([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H], outline='lime', width=3)

fig, ax = plt.subplots(1, 4, figsize=(22, 6))
ax[0].imshow(rgb); ax[0].set_title('1 raw 3D')
ax[1].imshow(ai_bg); ax[1].set_title(f'3 Qwen edit ({t_edit:.0f}s)')
ax[2].imshow(comp); ax[2].set_title('4 composite (vehicle frozen)')
ax[3].imshow(final_pil); ax[3].set_title('5 polished + bbox')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# # ⚖️ APPROVE GATE — divись на composite вище
#
# - ✓ Vehicle pixels intact (assert_pixel_identity не впав)?
# - ✓ Яскравість/тон техніки збігається з фоном (relight match_color)?
# - ✓ Фон не «пластиковий», виглядає як зйомка з дрона (не кіно)?
# - ✓ Масштаб рослинності відповідає aerial висоті?
# - ✓ Техніка на дорозі/колії, а не посеред поля?
# - ✓ Bbox tight навколо vehicle?
#
# **OK** → Cell 12 (5 frames).
#
# **НЕ ОК** → у SETTINGS (Cell 2) підкрути → re-run cell 2 → re-run Cell 10:
# - Фон «пластик»/оверсатур → `true_cfg_scale` 4 → 3.0 → 2.5
# - Vehicle іншої яскравості → `relight_strength` 0.55 → 0.7
# - Занадто «чисто/кіно» → підняти `POLISH['iso_noise']['p']`, знизити `jpeg.quality`
# - AI bg blurred → `steps` 25 → 35-40

In [ ]:
# Cell 12 — (опц) scale до 5 frames. Орієнтовно 15-30 хв на T4.
import time, os, subprocess, json
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

REPO_DIR = Path('/kaggle/working/yolo-bluebierd')
ASSETS = Path([p for p in Path('/kaggle/input').rglob('hdri')][0].parent)
OUTB = Path('/kaggle/working/v2_qwen_smoke_5frame')
OUTB.mkdir(parents=True, exist_ok=True)
CFG = Path('/kaggle/working/v2_qwen_smoke_1frame/v1_light_vehicle_qwen.yaml')

env = os.environ.copy(); env['MPLBACKEND'] = 'agg'
t_wall = time.time()
subprocess.run(['blenderproc', 'run',
                str(REPO_DIR/'datasetforge/engine/render_runner.py'),
                '--config', str(CFG), '--n', '5', '--out', str(OUTB),
                '--assets-root', str(ASSETS),
                '--seed', '100'], env=env, check=True)

for sub in ('ai_bg', 'composite', 'final'):
    (OUTB/sub).mkdir(exist_ok=True)
stems = sorted(p.stem for p in (OUTB/'images').glob('*.jpg'))
t0 = time.time()
for stem in stems:
    stage3_fn(pipe, rgb_path=OUTB/'images'/f'{stem}.jpg', depth_path=OUTB/'depth'/f'{stem}.png',
              mask_path=OUTB/'vehicle_masks'/f'{stem}.png', meta_path=OUTB/'metadata'/f'{stem}.json',
              out_path=OUTB/'ai_bg'/f'{stem}.png', diffusion_cfg=diff_cfg)
    composite.composite_one(rgb_path=OUTB/'images'/f'{stem}.jpg', ai_bg_path=OUTB/'ai_bg'/f'{stem}.png',
              mask_path=OUTB/'vehicle_masks'/f'{stem}.png', normals_path=OUTB/'normals'/f'{stem}.png',
              meta_path=OUTB/'metadata'/f'{stem}.json', out_path=OUTB/'composite'/f'{stem}.png',
              diffusion_cfg=diff_cfg, assert_pixel_identity=True)
    _s = int(json.loads((OUTB/'metadata'/f'{stem}.json').read_text()).get('seed', 0)) + int(polish_cfg.get('seed_offset', 9000))
    polish.polish_one(OUTB/'composite'/f'{stem}.png', OUTB/'final'/f'{stem}.png', polish_cfg, seed=_s)
wall = time.time() - t_wall
print(f'\n✓ {len(stems)} frames Stage 3+4+5 in {time.time()-t0:.0f}s (wall {wall:.0f}s)')

fig, ax = plt.subplots(len(stems), 3, figsize=(15, 5*len(stems)))
if len(stems) == 1: ax = ax[None, :]
for r, stem in enumerate(stems):
    ax[r,0].imshow(np.array(Image.open(OUTB/'images'/f'{stem}.jpg'))); ax[r,0].set_title(f'{stem} raw')
    ax[r,1].imshow(np.array(Image.open(OUTB/'ai_bg'/f'{stem}.png'))); ax[r,1].set_title('Qwen edit')
    ax[r,2].imshow(np.array(Image.open(OUTB/'final'/f'{stem}.png'))); ax[r,2].set_title('final (polished)')
    for c in range(3): ax[r,c].axis('off')
plt.tight_layout(); plt.show()